# Delay Target Analysis

Understanding the target variable `arrival_delay` before diving into sub-analyses.
Distribution, OTP baseline, arr vs dep comparison, cancellations.

## Setup

In [ ]:
from zh_tram_flow.notebook import *

TRAIN, TEST, lf = setup_analysis("03_analysis_target")

%load_ext autoreload
%autoreload 2

## Target Definition

**Primary target:** `arrival_delay` — seconds a tram arrives late at a stop (negative = early).

Chosen over `departure_delay` because it represents the passenger experience at every stop:
waiting time is determined by when the tram *arrives*, not when it leaves.

| Column | Role | Note |
|:---|:---|:---|
| `arrival_delay` | **Primary target** | Seconds late at arrival — what passengers experience |
| `departure_delay` | Feature | Seconds late at departure — starting condition for next leg |
| `delay_delta` | Derived feature | `departure_delay - arrival_delay` — positive = delay grows at stop, negative = delay recovered |
| `is_recovering` | Derived feature | `delay_delta < 0` — is the trip actively reducing its delay? |

**What we cannot observe:** whether a delay accumulated over multiple stops was fully recovered later.
`is_recovering` gives us a per-stop signal, but no trip-level recovery view.

In [ ]:
section_header("Derived delay features")

lf_target = lf.with_columns([
    (pl.col("departure_delay") - pl.col("arrival_delay")).alias("delay_delta"),
    ((pl.col("departure_delay") - pl.col("arrival_delay")) < 0).alias("is_recovering"),
])

## Delay Distribution

Overall shape of `arrival_delay`: min, max, mean, median, skewness. Foundation for all further analysis.

In [ ]:
section_header("Delay Distribution")

stats = (
    lf_target.select([
        pl.col("arrival_delay").min().alias("min"),
        pl.col("arrival_delay").max().alias("max"),
        pl.col("arrival_delay").mean().alias("mean"),
        pl.col("arrival_delay").median().alias("median"),
        pl.col("arrival_delay").std().alias("std"),
    ])
    .collect()
)
log(stats.to_pandas().T.to_string())

## On-Time Performance (OTP)

Share of trips arriving within ±120 seconds of schedule. Industry standard KPI for public transit.

In [ ]:
section_header("On-Time Performance")

otp = (
    lf_target
    .select([
        (pl.col("arrival_delay").abs() <= 120).mean().alias("otp_rate"),
        (pl.col("arrival_delay") > 120).mean().alias("late_rate"),
        (pl.col("arrival_delay") < -120).mean().alias("early_rate"),
    ])
    .collect()
)
log(f"On-Time  (|delay| ≤ 120s): {otp['otp_rate'][0]:.1%}")
log(f"Late     (delay  > 120s): {otp['late_rate'][0]:.1%}")
log(f"Early    (delay  < -120s): {otp['early_rate'][0]:.1%}")

## Arrival vs Departure Delay

Comparing `arrival_delay` and `departure_delay` distributions. Are stops adding or absorbing delay on average?

In [ ]:
section_header("Arrival vs Departure Delay")

comparison = (
    lf_target.select([
        pl.col("arrival_delay").mean().alias("arr_mean"),
        pl.col("departure_delay").mean().alias("dep_mean"),
        pl.col("delay_delta").mean().alias("delta_mean"),
        pl.col("is_recovering").mean().alias("recovering_share"),
    ])
    .collect()
)
log(comparison.to_pandas().T.to_string())

## Cancellations

`canceled = True` rows are the extreme case: effectively infinite delay. How many, when, where?

In [ ]:
section_header("Cancellations")

cancellations = (
    lf_target
    .group_by("is_canceled")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / pl.col("count").sum()).alias("share"))
    .sort("is_canceled")
    .collect()
)
log(cancellations.to_pandas().to_string())